### 06 — Train CNN Classifier

#### Objective

Train the custom ProductDefectCNN using real casting-product
images.

This notebook covers:

- loading prepared image DataLoaders
- dynamically determining the class count
- initializing the CNN
- configuring CrossEntropyLoss
- configuring the Adam optimizer
- training convolutional filters through backpropagation
- monitoring training and validation performance
- restoring the best validation model

The original test dataset is not used during training or model
selection.

##### 1 - Imports

In [0]:
import random
import mlflow
import mlflow.pytorch
import numpy as np
import torch
import torch.nn as nn

from src.cnn_model import ProductDefectCNN
from src.data_preparation import create_image_dataloaders
from src.model_training import train_model

from src.project_config import (
    LEARNING_RATE,
    NUM_EPOCHS,
    RANDOM_SEED,
    MLFLOW_EXPERIMENT_NAME,
    BATCH_SIZE,

)

In [0]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print(
    "MLflow experiment:",
    MLFLOW_EXPERIMENT_NAME,
)

##### 2 - Reproducibility

In [0]:
random.seed(
    RANDOM_SEED
)

np.random.seed(
    RANDOM_SEED
)

torch.manual_seed(
    RANDOM_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )

##### 3 - Device

In [0]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "PyTorch version:",
    torch.__version__,
)

print(
    "Training device:",
    device,
)

##### 4 - Load the real dataset

In [0]:
data = create_image_dataloaders()

In [0]:
print(
    "Class names:",
    data.class_names,
)

print(
    "Class mapping:",
    data.class_to_idx,
)

print(
    "Number of classes:",
    data.num_classes,
)

print(
    "Training images:",
    len(data.train_dataset),
)

print(
    "Validation images:",
    len(data.val_dataset),
)

print(
    "Test images:",
    len(data.test_dataset),
)

##### 5 - Validate one real batch

In [0]:
images, labels = next(iter(data.train_loader))

print(
    "Image batch shape:",
    images.shape,
)

print(
    "Label batch shape:",
    labels.shape,
)

print(
    "Image dtype:",
    images.dtype,
)

print(
    "Label dtype:",
    labels.dtype,
)

##### 6 - Build the real CNN

In [0]:
model = ProductDefectCNN(num_classes=data.num_classes)
model = model.to(device)
print(model)

##### 7. Verify real forward pass

In [0]:
#Before training thousands of images, validate the complete connection:

model.eval()

with torch.no_grad():

    sample_images = images.to(device)

    sample_logits = model(sample_images)

print(
    "Input shape:",
    sample_images.shape,
)

print(
    "Logits shape:",
    sample_logits.shape,
)

##### 8. Define loss

In [0]:
criterion = nn.CrossEntropyLoss()

##### 9. Define optimizer

In [0]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

##### 10. Verify trainable parameters

In [0]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(
    f"Total parameters: "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_parameters:,}"
)

Because this is a custom CNN:

Total parameters = Trainable parameters

All weights are being learned from our casting images.

##### 11. Train

Start MLflow before training

Instead of training first and opening an MLflow run afterward, put the training itself inside the MLflow run:

In [0]:
with mlflow.start_run(
    run_name="custom_cnn_training"
) as run:

    mlflow.log_params(
        {
            "model_type": "custom_cnn",
            "num_classes": data.num_classes,
            "class_names": str(data.class_names),
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "num_epochs": 5, #NUM_EPOCHS,
            "optimizer": "Adam",
            "loss_function": "CrossEntropyLoss",
        }
    )

    history = train_model(
        model=model,
        train_loader=data.train_loader,
        val_loader=data.val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=5,
    )

    for epoch in range(NUM_EPOCHS):

        mlflow.log_metrics(
            {
                "train_loss":
                    history["train_loss"][epoch],

                "train_accuracy":
                    history["train_accuracy"][epoch],

                "val_loss":
                    history["val_loss"][epoch],

                "val_accuracy":
                    history["val_accuracy"][epoch],
            },
            step=epoch + 1,
        )

    best_epoch_index = min(
        range(NUM_EPOCHS),
        key=lambda index:
            history["val_loss"][index],
    )

    best_epoch = best_epoch_index + 1

    best_val_loss = (
        history["val_loss"][best_epoch_index]
    )

    best_val_accuracy = (
        history["val_accuracy"][best_epoch_index]
    )

    mlflow.log_metrics(
        {
            "best_val_loss": best_val_loss,
            "best_val_accuracy":
                best_val_accuracy,
            "best_epoch": best_epoch,
        }
    )

    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="model",
    )

    cnn_run_id = run.info.run_id

print(
    "\nCNN MLflow Run ID:",
    cnn_run_id,
)

print(
    "Best epoch:",
    best_epoch,
)

print(
    "Best validation loss:",
    f"{best_val_loss:.4f}",
)

print(
    "Validation accuracy at best epoch:",
    f"{best_val_accuracy:.4f}",
)

print(
    "Trained CNN successfully persisted to MLflow."
)

This is a smoke test, not the real experiment. Confirm that one complete training + validation epoch executes successfully and see how long it takes.

If that works, recreate the model and optimizer from scratch before the real run, then use NUM_EPOCHS. That avoids starting the final experiment from weights already modified by the smoke test.

Once you have the Epoch 1 output and runtime, verify the loss/accuracy behavior before committing CPU time to the full training run.

In [0]:
training_result  = train_model(
    model=model,
    train_loader=data.train_loader,
    val_loader=data.val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    #num_epochs=NUM_EPOCHS,
)

In [0]:
history = (
    training_result.history
)

In [0]:
print("History keys:", history.keys())

for key, values in history.items():
    print(
        key,
        "length:",
        len(values),
        "values:",
        values,
    )

In [0]:
print("NUM_EPOCHS:", NUM_EPOCHS)
print("Actual epochs in history:", len(history["train_loss"]))

In [0]:
num_completed_epochs = len(history["train_loss"])

In [0]:
import mlflow
import mlflow.pytorch

# Number of epochs that actually completed
num_completed_epochs = len(
    history["train_loss"]
)

print(
    "Completed epochs:",
    num_completed_epochs,
)

# Find the best epoch from the completed training
best_epoch_index = min(
    range(num_completed_epochs),
    key=lambda index:
        history["val_loss"][index],
)

best_epoch = best_epoch_index + 1

best_val_loss = (
    history["val_loss"][best_epoch_index]
)

best_val_accuracy = (
    history["val_accuracy"][best_epoch_index]
)


with mlflow.start_run(
    run_name="custom_cnn_training"
) as run:

    # Log training configuration
    mlflow.log_params(
        {
            "model_type": "custom_cnn",
            "num_classes": data.num_classes,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "num_epochs": num_completed_epochs,
            "optimizer": "Adam",
            "loss_function": "CrossEntropyLoss",
        }
    )

    # Log metrics for every completed epoch
    for epoch in range(num_completed_epochs):

        mlflow.log_metrics(
            {
                "train_loss":
                    history["train_loss"][epoch],

                "train_accuracy":
                    history["train_accuracy"][epoch],

                "val_loss":
                    history["val_loss"][epoch],

                "val_accuracy":
                    history["val_accuracy"][epoch],
            },
            step=epoch + 1,
        )

    # Log best validation results
    mlflow.log_metrics(
        {
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "best_val_accuracy":
                best_val_accuracy,
        }
    )

    # SAVE THE TRAINED MODEL
    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="model",
    )

    cnn_run_id = run.info.run_id


print("\nCNN MLflow Run ID:", cnn_run_id)
print("Best epoch:", best_epoch)
print(
    f"Best validation loss: "
    f"{best_val_loss:.4f}"
)
print(
    f"Best validation accuracy: "
    f"{best_val_accuracy:.4f}"
)
print(
    "Best CNN successfully persisted to MLflow."
)

##### 12. What is happening inside that one line?

This: history = train_model(...)

looks simple because we moved reusable logic into src.

Internally:

``` text

Epoch 1
  ↓
Batch 1
  ↓
32 casting images
  ↓
CNN forward
  ↓
2 logits/image
  ↓
CrossEntropyLoss
  ↓
zero_grad()
  ↓
backward()
  ↓
Conv filter gradients
+
classifier gradients
  ↓
Adam.step()
  ↓
weights updated

Batch 2
  ↓
repeat

...

all 5,638 training images
  ↓
TRAIN epoch complete

995 validation images
  ↓
forward only
  ↓
validation loss + accuracy
  ↓
No backprop
  ↓
No optimizer update

        ↓

Epoch 2
        ↓
repeat

```

That's the real CNN learning process.

##### 13. Inspect training history

In [0]:
print(
    "Training loss:",
    history["train_loss"],
)

print(
    "Validation loss:",
    history["val_loss"],
)

print(
    "Training accuracy:",
    history["train_accuracy"],
)

print(
    "Validation accuracy:",
    history["val_accuracy"],
)

We'll later visualize these properly.

For now, the main things we'll watch are:

Train loss ↓
Validation loss ↓

Train accuracy ↑
Validation accuracy ↑

But they won't necessarily improve smoothly every epoch.

##### 14. Check final restored model

In [0]:
#Remember: after train_model() finishes, model contains the weights from the lowest validation-loss epoch, not necessarily the last epoch.


model.eval()

with torch.no_grad():

    final_logits = model(sample_images)

    final_predictions = torch.argmax(final_logits, dim=1,)

print(
    "Sample predictions:",
    final_predictions[:10].cpu(),
)

print(
    "Actual labels:",
    labels[:10],
)

##### 15. Optional probabilities for inspection

In [0]:
with torch.no_grad():

    probabilities = torch.softmax(
        final_logits,
        dim=1,
    )

print(
    "First image logits:",
    final_logits[0].cpu(),
)

print(
    "First image probabilities:",
    probabilities[0].cpu(),
)

##### 16. Do NOT evaluate the test set yet

Even though: data.test_loader

exists, don't calculate test accuracy in Notebook 06.

That's intentional.

``` text

TRAIN
↓
learn parameters

VALIDATION
↓
monitor/model selection

TEST
↓
leave untouched

```
Notebook:

07_evaluate_cnn_classifier

will perform the final test evaluation.

That separation is good experimental discipline.

##### 17. 🔍 XAI connection

At the beginning of training:

CNN filters
≈ initialized numerical weights

After training:
``` text

CNN filters
↓
learned through classification loss
↓
useful visual representations

```

But training only tells the model:

Reduce classification error.

It doesn't explicitly tell the model:

Look only at the actual casting defect.

Later, 🔍 XAI will help us investigate what image regions contribute to predictions.

##### 18. ⚖️ RAI connection

We already discovered:

Train source:

def_front  = 3,758
ok_front   = 2,875

and:

Test:

def_front = 453
ok_front  = 262

Therefore Notebook 07 should not simply say:

Accuracy = X%

We'll examine class-specific behavior.

For an industrial inspection system, particularly interesting cases include:

Actual:
def_front

Predicted:
ok_front

because that represents a defective casting being classified as acceptable.

Whether that error has greater operational significance depends on the intended deployment context, but it is exactly the kind of failure mode we should explicitly analyze under ⚖️ RAI reliability/safety.

##### ## Key Learnings

1. The custom CNN is trained using real casting-product images.

2. The number of output classes is derived dynamically from the
   dataset rather than hard-coded.

3. The real classification task contains two classes:
   def_front and ok_front.

4. The CNN produces two raw logits for each image.

5. CrossEntropyLoss compares the raw logits with integer class
   labels.

6. Softmax is not manually applied before CrossEntropyLoss.

7. optimizer.zero_grad() clears gradients from the previous
   training batch.

8. loss.backward() computes gradients throughout the complete
   network.

9. Convolution filter weights receive gradients and are learned
   through backpropagation.

10. optimizer.step() updates both convolutional and classifier
    parameters.

11. One epoch represents one complete pass through the training
    dataset.

12. Validation data is used to monitor generalization without
    updating model parameters.

13. model.train() and model.eval() control model behavior but do
    not themselves perform training or evaluation.

14. The model state with the lowest validation loss is restored
    after training.

15. The original test set remains untouched during training and
    model selection.

16. 🔍 XAI will later help determine whether the trained CNN is
    focusing on meaningful defect regions.

17. ⚖️ RAI requires examining important failure modes and
    class-specific performance rather than relying only on
    aggregate accuracy.

In [0]:
import mlflow
import mlflow.pytorch

from src.project_config import (
    BATCH_SIZE,
    LEARNING_RATE,
    NUM_EPOCHS,
)

with mlflow.start_run(
    run_name="custom_cnn_5_epoch_training"
) as run:

    mlflow.log_params(
        {
            "model_type": "custom_cnn",
            "num_classes": data.num_classes,
            "learning_rate": LEARNING_RATE,
            "num_epochs": NUM_EPOCHS,
            "batch_size": BATCH_SIZE,
        }
    )

    mlflow.log_metrics(
        {
            "best_val_loss": 0.3589,
            "best_val_accuracy": 0.8422,
        }
    )

    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="model",
    )

    cnn_run_id = run.info.run_id

print(
    "CNN MLflow Run ID:",
    cnn_run_id,
)